# Background
This is my simplified implementation of the compara PTM conservation pipeline

What it does:
- Downloads compara files and separates into folder structure
- Extracts sequence alignments for proteins with PTMs and desired species
- Extracts PTM/PTMacceptor stats at alignment positions, optionally using a window around the PTM site
- Combines results

What it outputs:
- Results for individual gene families are stored in a nested set of directories called `temp/ENSTREE_X` where `X` is the ensemble version. Each directory contains:
  - `ENSTREE_X.aa.fasta`: The entire sequence alignment from compara
  - `ENSTREE_X.gnt`: Mapping between species and genes in the alignment
  - `ENSTREE_X.list`: Mapping between species and genes in the alignment
  - `{speciestag}/ENSTREE_X_reduced.aa.nogap.fasta`: The alignment reduced to species of interest, with empty columns removed
  - `{speciestag}/window/ENSTREE_X_ptm_continue_region_{window}.txt`: PTM statistics for the alignment. For each alignment column with a PTM, this assigns a score of 1.0 to genes with a PTM, 0.5 to genes with a PTM acceptor residue, and 0.0 otherwise
  - `{speciestag}/window/ENSTREE_X_ptm_continue_region_{window}_origins.txt`: A table containing all PTM sites for this gene family, including the species and a map between the site position and the alignment column
- `results/all_origins_{speciestag}_{window}.tab`: A combined table of all PTM sites with a map between site position and the alignment column, derived from the individual `ENSTREE_X_ptm_continue_region_{window}_origins.txt` files

Dependencies:
- python > 3.9
- pandas 2.2.3 (probably doesn't need to be this specific version, this is just what I used)
- biopython 1.85 (probably doesn't need to be this specific version, this is just what I used)

In [ ]:
# Setup and Configuration
import os
import subprocess
import shutil
import gzip
from pathlib import Path
import sys
import pandas as pd
from Bio import SeqIO
import urllib

# Mount the src directory so I can load scripts from there
import sys
SRCDIR = "src"
sys.path.insert(0, str(SRCDIR))
from main import remove_all_gap_columns
from main import create_position_mapping
from main import findPtmColumns
from main import calculatePtmScores
from main import writePtmContinueFiles
from main import get_species_name_fallback

# Configuration variables (equivalent to Makefile variables)
COMPARARELEASE = 86
SPECIESTAG = "PRIDE"
PTMMODE = "ubi"  # other option: "phospho"
SAMPLESIZE = 4000
REGIONWINDOW = 0

# Directory paths
ROOTDIR = Path.cwd()
DATADIR = ROOTDIR / "data"
TEMPDIR = ROOTDIR / "temp"
SRCDIR = ROOTDIR / "src"
RESULTSDIR = ROOTDIR / "results"

# Input files
SPECIESFILE = DATADIR / "species_list.txt"  # List of species names to be considered
ALLSITES = DATADIR / "all_sites_ptmdb.tab"  # All PTM sites

# Key output files
COMPARAFASTA = TEMPDIR / f"Compara.{COMPARARELEASE}.protein.aa.fasta"
COMPARATREES = TEMPDIR / f"Compara.{COMPARARELEASE}.protein.nh.emf"
GENETOTREEFILE = TEMPDIR / "Gene_to_tree_file.txt"
GENETOTREEHUMANFILE = TEMPDIR / "Gene_to_tree_file_human.txt"
GENETREEDIR = TEMPDIR / f"ENSTREE_{COMPARARELEASE}"

# URLs for downloads
COMPARAFTPFASTA = f"ftp://ftp.ensembl.org/pub/release-{COMPARARELEASE}/emf/ensembl-compara/homologies/Compara.{COMPARARELEASE}.protein.aa.fasta.gz"
COMPARAFTPTREE = f"ftp://ftp.ensembl.org/pub/release-{COMPARARELEASE}/emf/ensembl-compara/homologies/Compara.{COMPARARELEASE}.protein.nh.emf.gz"
NCBITAXONOMYFTP = "ftp://ftp.ncbi.nlm.nih.gov/pub/taxonomy/taxdump.tar.gz"

print("Configuration complete!")
print(f"Working directory: {ROOTDIR}")
print(f"Compara release: {COMPARARELEASE}")
print(f"Species tag: {SPECIESTAG}")
print(f"PTM mode: {PTMMODE}")

In [ ]:
def run_script(script_path, args, stdout_file=None, verbose=False):
    """Run a Python script with arguments"""
    cmd = [sys.executable, str(script_path)] + [str(arg) for arg in args]

    if verbose:
        print(f"Running: {' '.join(cmd)}")
    
    try:
        if stdout_file:
            with open(stdout_file, 'w') as f:
                process = subprocess.Popen(
                    cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                    text=True, bufsize=1, universal_newlines=True
                )
                
                for line in process.stdout:
                    if verbose:
                        print(line.rstrip())
                    f.write(line)
                
                process.wait()
                if process.returncode != 0:
                    stderr_output = process.stderr.read()
                    if verbose:
                        print(f"Error: {stderr_output}")
                    return False
                return True
        else:
            process = subprocess.Popen(
                cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                text=True, bufsize=1, universal_newlines=True
            )
            
            for line in process.stdout:
                if verbose:
                    print(line.rstrip())
            
            process.wait()
            if process.returncode != 0:
                stderr_output = process.stderr.read()
                if verbose:
                    print(f"Error: {stderr_output}")
                return False
            return True
            
    except Exception as e:
        if verbose:
            print(f"Error running script: {e}")
        return False

In [ ]:
# Create directories
def create_directories():
    """Create all necessary directories"""
    directories = [
        TEMPDIR, DATADIR, RESULTSDIR
    ]
    
    for directory in directories:
        directory.mkdir(parents=True, exist_ok=True)
        print(f"Created directory: {directory}")

create_directories()

## Make intermediate files

In [ ]:
# List of input species
## These are kept in alignments
species_list = []
species_file = open(SPECIESFILE, "r")
while 1:
    line = species_file.readline()
    if line == "":
        break
    species = line.rstrip()
    species_list.append(species.replace(" ", "_"))
species_file.close()
print(species_list)

# Load in ptm sites
allSites = pd.read_csv(ALLSITES, sep = "\t")

# Dicitonary of allSites
allGenes = sorted(list(set(allSites["ensp"])))
allSitesDct = {key: [] for key in allGenes}
for _, row in allSites.iterrows():
    allSitesDct[row["ensp"]].append(str(row["position"]))

# PTM acceptors
ptm_acceptors = {
        "ubi": ["K"],                    # Ubiquitination occurs on lysine
        "phospho": ["S", "T", "Y"]       # Phosphorylation occurs on serine, threonine, tyrosine
    }

## Step 1: Download Compara Data

Download Ensembl Compara FASTA and tree files. These contain protein sequences and phylogenetic trees for all gene families across species.

In [ ]:
def download_and_extract(url, output_path):
    """Download a gzipped file and extract it"""
    if output_path.exists():
        print(f"File already exists: {output_path}")
        return
    
    gz_path = output_path.with_suffix(output_path.suffix + '.gz')
    
    print(f"Downloading {url}...")
    try:
        urllib.request.urlretrieve(url, gz_path)
        print(f"Downloaded to {gz_path}")
        
        print("Extracting...")
        with gzip.open(gz_path, 'rb') as f_in:
            with open(output_path, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        
        os.remove(gz_path)  # Clean up
        print(f"Extracted to {output_path}")
        
    except Exception as e:
        print(f"Error downloading {url}: {e}")

# Download Compara FASTA
download_and_extract(COMPARAFTPFASTA, COMPARAFASTA)

In [ ]:
# Download Compara Trees
download_and_extract(COMPARAFTPTREE, COMPARATREES)

# Step 2: Process Compara Files

Extract individual gene family alignments and trees from the large Compara files using the existing Python scripts.

**Warning:** These steps are slow

## Compara AA file
Split the compara Compara.XX.protein.aa.fasta file, which contains the AA alignments, into individual files for each gene family

In [ ]:
# Step 2.1: Generate alignments
if False:   # Toggle whether this step is run
    print("Generating alignments...")
    success = run_script(SRCDIR / "ensembl2aafasta.py", [COMPARAFASTA, GENETREEDIR])
    if success:
        print("Alignments generated successfully")
    else:
        print("Failed to generate alignments")

## Compara gnt file
Split the compara Compara.XX.protein.nh.emf file, which contains the species-level data on gene families, into individual files. Also create a master sheet `Gene_to_tree_file.txt`, which stores the directories for each gene and its corresponding gene family

In [ ]:
# Step 2.2: Generate gene trees and mapping file
if False:  # Toggle whether this step is run
    print("Generating gene trees...")
    success = run_script(SRCDIR / "ensembl2gnt.py", [COMPARATREES, GENETREEDIR], GENETOTREEFILE)
    if success:
        print(" Gene trees generated successfully")
        print(f" Gene-to-tree mapping saved to {GENETOTREEFILE}")
    else:
        print("Failed to generate gene trees")


# Check the results
if GENETOTREEFILE.exists():
    with open(GENETOTREEFILE) as f:
        lines = f.readlines()
    print(f"Generated {len(lines)} gene-to-tree mappings")
    print("First few lines:")
    for line in lines[:5]:
        print(f"  {line.strip()}")

## Subset gene mapping for families with human genes
We create a smaller master file, `Gene_to_tree_file_human.txt`, which stores the file locations for all human genes and their gene families

In [ ]:
try:
    with open(GENETOTREEFILE) as infile, open(GENETOTREEHUMANFILE, 'w') as outfile:
        human_genes = 0
        for line in infile:
            if line.startswith("ENSP0"):  # Human genes start with ENSP0
                outfile.write(line)
                human_genes += 1
    
    print(f"Created human gene mapping with {human_genes} genes")
except Exception as e:
    print(f"Error creating human gene mapping: {e}")

# Step 3: Process alignments with PTMs

## Get all gene families with a PTM and prepare

In [ ]:
# Load in Gene-path mastersheet
GeneToTreeHuman = pd.read_csv(GENETOTREEHUMANFILE, sep="\t", header=None)
GeneToTreeHuman.columns = ["gene", "family", "path"]
GeneToTree = pd.read_csv(GENETOTREEFILE, sep="\t", header=None)
GeneToTree.columns = ["gene", "family", "path"]

# Correct  directories (this is user specific)
CORRECTIONTAG = ""
GeneToTreeHuman["path"] = GeneToTreeHuman["path"].str.replace(
    CORRECTIONTAG, ""
)
GeneToTree["path"] = GeneToTree["path"].str.replace(
    CORRECTIONTAG, ""
)

# Subset for trees with a ptm 
GeneToTree_wPTM = GeneToTree[GeneToTree["gene"].isin(allSites["ensp"])]
## Make unique and sort
families_wPTM = sorted(list(set(GeneToTree_wPTM["family"]))) 
## Get paths in a dictionary
families_wPTM_pathDct = {}
distinct_fam_path = GeneToTree_wPTM[['family', 'path']].drop_duplicates()
for _, row in distinct_fam_path.iterrows():
    families_wPTM_pathDct[row["family"]] = row["path"]

# Set up output file containing PTM sites and their column-position mapping
df_origins_lst = []

## Loop over gene families and process alignments individually

In [ ]:
# Set up gene families
keys = list(families_wPTM_pathDct.keys())

# Loop
for i, family in enumerate(keys):

    if (i % 100 == 0):
        print("Processed " + str(i) + " entries")

    path = families_wPTM_pathDct[family]
    
    ##### Reduce alignment to species of interest #####
    # Make new path for output
    subpath = path+"/"+ SPECIESTAG
    Path(subpath).mkdir(parents=True, exist_ok=True)
    #os.chdir(subpath)

    # Get files to load
    aln_file = path + "/" + family + ".aa.fasta" #Seuqence alignment
    gene_file = path + "/" + family + ".list" #Map of genes to species

    # Load in genes and their species, and subset for genes of interest
    # In the process, also make a gene-to-species dictionary
    with open(gene_file) as fin:
        reduced_gene_species_dct = {
            line.split("\t")[0]: line.strip().split("\t")[1].replace(" ", "_")
            for line in fin
            if line.strip().split("\t")[1].replace(" ", "_") in species_list
        }
    # Convert "genus_species" to "Genus species" format
    for gene, species_code in reduced_gene_species_dct.items():
        if "_" in species_code:
            genus, species = species_code.split("_", 1)
            species_name = f"{genus.capitalize()} {species.lower()}"
            reduced_gene_species_dct[gene] = species_name
    # Get genes
    reduced_gene_set = list(reduced_gene_species_dct.keys())
        
    # Remove empty first line
    def remove_leading_blank_lines(infile, outfile):
        with open(infile) as f, open(outfile, "w") as out:
            started = False
            for line in f:
                if not started and not line.strip():
                    # skip leading blanks
                    continue
                started = True
                out.write(line)
    aln_file_cleaned = aln_file.replace(".aa", "clean.aa")
    remove_leading_blank_lines(aln_file, aln_file_cleaned)

    # Reduce alignment by filtering records and writing output using SeqIO.write
    with open(aln_file_cleaned) as handle:
        aln_cleaned = [record for record in SeqIO.parse(handle, "fasta") if record.id in reduced_gene_set]

    ##### Remove empty columns in the alignment #####
    nogap_aln_file = subpath + "/"  + family + "_reduced.aa.nogap.fasta"
    aln_nogap = remove_all_gap_columns(aln_cleaned, nogap_aln_file)  # From the script scr/main



    ##### Process sequences, make dictionary of positions, etc #####

    # Convert sequences to dictionary
    sequences = {}
    for record in aln_nogap:
        sequences[record.id] = str(record.seq)

    # Create position mappings for all sequences
    position_mappings = {}
    for gene_id in sequences:
        position_mappings[gene_id] = create_position_mapping(sequences[gene_id])  # From the script src/main

    # Find alignment columns containing PTM sites
    ptm_columns = findPtmColumns(sequences, position_mappings, allSitesDct, ptm_acceptors[PTMMODE])   # From the script src/main


    ##### Score alignment positions for each gene based on presenceof PTM or acceptor #####

    # Initialize scoring dictionary to track PTM scores
    # Score = 1.0 for ptms, and 0.5 for PTM acceptors
    scoring_dict = {}
    # Analyze each PTM column
    for column in ptm_columns:
        # Calculate scores for all genes at this column for ancestral reconstruction
        for gene_id in sequences:
            ptm_score = calculatePtmScores(sequences, position_mappings, allSitesDct, ptm_acceptors[PTMMODE], gene_id, REGIONWINDOW, column) # From the script src/main
            
            # Store score in scoring dictionary
            if gene_id not in scoring_dict:
                scoring_dict[gene_id] = {}
            scoring_dict[gene_id][column] = ptm_score

    # Write ptm_continue_region files containing scores at each position
    writePtmContinueFiles(family, GENETREEDIR, SPECIESTAG, REGIONWINDOW, ptm_columns, scoring_dict, sequences) # From the script src/main

    ##### Export list of PTM sites and their column-position mappings
    # Calculate directory structure and create output directory
    tree_number = int(family.split("_")[1])
    supra_folder = f"ENSTREE_{str(int(str(tree_number).zfill(5)[0:2]) + 1).zfill(2)}000"
    output_dir = Path(GENETREEDIR) / supra_folder / family / SPECIESTAG / f"region_w{REGIONWINDOW}"

    # Output file
    origins_file = output_dir / f"{family}_ptm_continue_region_w{REGIONWINDOW}_origins.txt"

    # Create file and write output
    with open(origins_file, 'w') as origins_out:
        # Collect rows for output as a list of records
        records = []
        
        for column in ptm_columns:
            for gene_id in sequences:
                if gene_id in allSitesDct and column in position_mappings[gene_id]:
                    # Get sequence position (1-based)
                    sequence_position = position_mappings[gene_id][column] + 1
                    
                    # Check if this position has a verified PTM site
                    if str(sequence_position) in allSitesDct[gene_id]:
                        # Get species name from gene-to-species mapping
                        species = reduced_gene_species_dct.get(gene_id, get_species_name_fallback(gene_id))  # From the script src/main
                        # Append record: family, column (1-based), species, gene, position
                        records.append([family, column + 1, species, gene_id, sequence_position])
        
        # Create a DataFrame and export to the output file as a TSV without header/index
        df_origins = pd.DataFrame(records, columns=["family", "column", "species", "gene", "position"])
        df_origins.to_csv(origins_file, sep="\t", index=False, header=False)
    # Add to master origins file
    df_origins_lst.append(df_origins)

# Export master origins file
df_origins_combined = pd.concat(df_origins_lst, ignore_index=True)
outDir = RESULTSDIR / f"all_origins_{SPECIESTAG}_w{REGIONWINDOW}.tab"
df_origins_combined.to_csv(outDir, index = False, sep = "\t")